In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [4]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [6]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          "That's "
                                                                          'fine, '
                                                                          "let's "
                                                                          'reschedule. '
                                                                          'What '
   

In [7]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, thanks for letting me know. That's fine, let's reschedule. What time works for you? Best, Seán."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, thanks for letting me know. That\'s fine, let\'s reschedule. What time works for you? Best, Seán."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='70974dff853caa3c5efd5744881e5805')]


In [8]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, thanks for letting me know. That's fine, let's reschedule. What time works for you? Best, Seán.


## Approve

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='56969f8a-f5a0-43c5-8c37-17e9caaa35af'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "inbox"}'}, '__gemini_function_call_thought_signatures__': {'call_1218351': 'EnEKbwFpFH0TZhOL8d8PFUzxZ5S5NTewlRCwqL4+StpYtA4xiSy5MTC9XBtf0rCR6m6Oi1KdWdS/VjhsmmpQ++pCpSria1zAexX1CPJYTh1tewykNOKIAcXv0dfrvksTVnJDBA1UyxISJxp1itXNFpcUow=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d944-102b-7bc0-b934-2f4d89314164-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'inbox'}, 'id': 'call_1218351', 'type': 'tool_call'}], invali

## Reject

In [10]:
from pprint import pprint
from langgraph.types import Command
from langchain.messages import HumanMessage

# Fresh thread: pause on a NEW pending send_email so the 'reject' decision has its own approval to refuse
config = {"configurable": {"thread_id": "2"}}
agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
    },
    config=config,
)

response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "No please sign off - Your merciful leader, Seán.",
                }
            ]
        }
    ),
    config=config,
)

pprint(response)


{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                    

In [11]:
from langchain.messages import ToolMessage

# Reject = send_email was NEVER executed: not a single "Email sent" in thread 2.
sent = [str(m.content) for m in response["messages"] if isinstance(m, ToolMessage)]
print('Email sent anywhere in this thread? ->', any('Email sent' in c for c in sent))
print('Final assistant answer:', str(response['messages'][-1].content)[:160])


Email sent anywhere in this thread? -> False
Final assistant answer: []


## Edit

In [12]:
from langchain.messages import AIMessage

# Fresh thread: pause again so the 'edit' decision can replace the tool's args
config = {"configurable": {"thread_id": "3"}}
agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
    },
    config=config,
)

response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {"body": "This is the last straw, you're fired!"},
                    },
                }
            ]
        }
    ),
    config=config,
)

pprint(response)

# Edit took effect: the args that were actually used
edited_call = [m for m in response["messages"] if isinstance(m, AIMessage) and m.tool_calls][-1]
print("Edited call args used:", edited_call.tool_calls[0]["args"])


{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='bae3f933-41e2-40ff-a677-59a0b52c657c'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "inbox"}'}, '__gemini_function_call_thought_signatures__': {'call_1357307': 'EnEKbwFpFH0TtumhOAQS/ec/HNktxgVGp7zopq31EgrGVO0JHwLGyZbli2xIr7hz5Y06GU2A9a6vCPayeL3EXgktA83pGqi7O6z5Zr0BOJ96plO2BjUropDqHh5AXpk4hJQ5MvQ/Ioz9tHbwSenOcdIPZQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d944-7b3d-7d12-b745-9651cb49a8bc-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'inbox'}, 'id': 'call_1357307', 'type': 'tool_call'}], invali